# Graph Attention Networks — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sigmoid(x): return 1/(1+np.exp(-x))
def softmax(x):
    x=np.asarray(x,dtype=float); e=np.exp(x-x.max()); return e/e.sum()
def draw_graph(A, pos=None, values=None, title=''):
    A=np.asarray(A); n=A.shape[0]
    if pos is None:
        ang=np.linspace(0,2*np.pi,n,endpoint=False); pos=np.c_[np.cos(ang),np.sin(ang)]
    plt.figure(figsize=(4,4))
    for i in range(n):
        for j in range(i+1,n):
            if A[i,j]!=0: plt.plot([pos[i,0],pos[j,0]],[pos[i,1],pos[j,1]],c='0.75',lw=1+2*abs(A[i,j]))
    c=values if values is not None else np.arange(n)
    plt.scatter(pos[:,0],pos[:,1],c=c,s=320,cmap='viridis',edgecolor='k',zorder=3)
    for i,(x,y) in enumerate(pos): plt.text(x,y,str(i),ha='center',va='center',color='white',weight='bold')
    plt.axis('off'); plt.title(title)
print('setup ok')

## Attention lets a node decide which neighbors matter most
A GAT replaces uniform averaging with learned edge weights. The examples compute logits, softmax coefficients, a weighted message, a second head, and a masked attention row that ignores non-neighbors.

In [ ]:
# 1) attention logits from query-key dot products
q=np.array([1.,0.]); K=np.array([[1.,0.],[0.,1.],[1.,1.]]); logits=K@q
plt.figure(figsize=(5,3)); plt.bar(['n0','n1','n2'],logits); plt.title('attention logits')
assert np.allclose(logits,[1,0,1])

In [ ]:
# 2) softmax turns logits into edge weights
alpha=softmax([1,0,1])
plt.figure(figsize=(5,3)); plt.bar(['n0','n1','n2'],alpha); plt.title('softmax attention coefficients')
assert np.allclose(alpha,[0.4223188,0.1553624,0.4223188])

In [ ]:
# 3) weighted neighbor message
V=np.array([[2.,0.],[0.,2.],[1.,1.]]); alpha=softmax([1,0,1]); msg=alpha@V
plt.figure(figsize=(5,3)); plt.bar(['feat0','feat1'],msg); plt.title('attention-weighted message')
assert np.allclose(msg,[1.2669564,0.7330436])

In [ ]:
# 4) multi-head attention averages different views
m1=np.array([1.2669564,0.7330436]); m2=np.array([0.8,1.2]); out=(m1+m2)/2
plt.figure(figsize=(5,3)); plt.bar(['feat0','feat1'],out); plt.title('two-head average')
assert np.allclose(out,[1.0334782,0.9665218])

In [ ]:
# 5) masking sets non-neighbor attention to zero
logits=np.array([2.,1.,5.]); mask=np.array([1,1,0],bool); a=np.zeros(3); a[mask]=softmax(logits[mask])
plt.figure(figsize=(5,3)); plt.bar(['nbr0','nbr1','non-nbr'],a); plt.title('masked graph attention')
assert a[2]==0 and abs(a[:2].sum()-1)<1e-9